# Features and Models: What Actually Moves the Number

**Computational Text Analysis Pack · Module 07 — Classification**

*Technique:* Comparing feature settings and classifiers; cross-validation

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Sagaustus/adh-dh-notebooks/blob/main/06-supervised-classification/07b_features_and_models.ipynb)

Data: `masakhanews.csv` — from the `data/` folder of this pack.

---

In [ ]:
# Setup — run this first.
import os, warnings
warnings.filterwarnings('ignore')
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# The data this notebook needs. It is downloaded from CourseHub automatically;
# uploading by hand is only a fallback if that server is unreachable.
NEEDED = ['masakhanews.csv']
DATA_URL = 'https://coursehub.africandigitalhumanities.ca/ide/pack-data/text/'

def _missing():
    return [f for f in NEEDED if not os.path.exists(f)]

missing = _missing()
if missing:
    # Fetch from CourseHub first, so nobody has to upload anything.
    import urllib.request, urllib.parse
    for _f in list(missing):
        try:
            urllib.request.urlretrieve(DATA_URL + urllib.parse.quote(_f), _f)
            print(f'downloaded {_f}')
        except Exception as _e:
            print(f'could not fetch {_f}: {_e}')
    missing = _missing()

if missing:
    try:
        from google.colab import files
    except ImportError:
        raise SystemExit('Place these next to the notebook: ' + ', '.join(missing))
    # Ask again until everything has arrived. The upload widget returns as soon
    # as you close it, so picking only some of the files would otherwise fail a
    # few lines below with a confusing FileNotFoundError.
    for _ in range(4):
        print('Select ALL of these at once (ctrl-click / cmd-click to multi-select):')
        print('   ' + ', '.join(missing))
        files.upload()
        missing = _missing()
        if not missing:
            break
        print('Still needed: ' + ', '.join(missing))
    if missing:
        raise SystemExit(
            'Missing: ' + ', '.join(missing) + '. Re-run this cell and select '
            'every file listed, or upload them with the folder icon on the left.')

df       = pd.read_csv('masakhanews.csv')
NIGERIAN = ['Hausa', 'Igbo', 'Yoruba', 'Nigerian Pidgin']
# Most of this pack runs on scikit-learn and the standard library. Also
# available: nltk, spacy, gensim and networkx (modules 14-19), Keras
# (case studies), and a GPU pod reached through /ide/api/neural/ for the
# neural modules — never called directly, so no token lives in a notebook.

plt.rcParams['figure.figsize'] = (10, 5.5)
plt.rcParams['axes.grid'] = True
plt.rcParams['grid.alpha'] = 0.25

print(f'Loaded. df has {len(df):,} rows and {df.shape[1]} columns.')

## What this notebook does

There is a temptation to try settings until accuracy looks good. That is how you
end up reporting a number that will not replicate.

This notebook runs the comparison properly, with **cross-validation** rather than
a single split, and shows the honest result: for text, feature choices matter
more than the choice of classifier, and most of the differences are inside the
noise.

## The tooling

**scikit-learn** end to end: `train_test_split`, `LogisticRegression`,
`LinearSVC`, `MultinomialNB`, `cross_val_score`. This is the library's core
competence and the reason it is the general practitioner of the Python text
stack.

In [ ]:
import re

def tokenise(text):
    """Words, lowercased, Unicode-aware.

    `\\w+` matches Unicode letters, so ọ, à, ƙ, ɗ and ɓ survive. The common
    shortcut `[a-z]+` would throw them away and quietly change every count in
    this notebook — see notebook 03b.
    """
    return re.findall(r"\w+", str(text).lower())

sample = "Àwọn ọmọ ìlú ti ṣe ìpàdé — ƙungiyar ɗalibai ta yi taro"
print("sample     :", sample)
print("tokens     :", tokenise(sample))
print("naive [a-z]:", re.findall(r"[a-z]+", sample.lower()))
print()
print("The naive pattern splits words at every accented letter.")

In [ ]:
from sklearn.model_selection import cross_val_score, StratifiedKFold
from sklearn.feature_extraction.text import TfidfVectorizer, CountVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import MultinomialNB
from sklearn.svm import LinearSVC
from sklearn.pipeline import make_pipeline

sub = df[df["language"] == "English"]
keep = sub["category"].value_counts()
sub = sub[sub["category"].isin(keep[keep >= 40].index)].reset_index(drop=True)
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=0)

print(f"{len(sub):,} documents, {sub['category'].nunique()} categories")
print("Everything below is 5-fold cross-validated: the whole pipeline is refitted")
print("inside each fold, so the vectoriser never sees the held-out documents.")

In [ ]:
def score(vectoriser, model, name):
    pipe = make_pipeline(vectoriser, model)
    s = cross_val_score(pipe, sub["text"], sub["category"], cv=cv, n_jobs=1)
    print(f"{name:<44}{s.mean():.3f} ± {s.std():.3f}")
    return s.mean(), s.std()

results = {}
results["counts + logistic regression"] = score(
    CountVectorizer(token_pattern=r"\w+", min_df=3, max_features=5000),
    LogisticRegression(max_iter=2000), "counts + logistic regression")
results["tf-idf + logistic regression"] = score(
    TfidfVectorizer(token_pattern=r"\w+", min_df=3, max_features=5000, sublinear_tf=True),
    LogisticRegression(max_iter=2000), "tf-idf + logistic regression")
results["tf-idf + naive Bayes"] = score(
    TfidfVectorizer(token_pattern=r"\w+", min_df=3, max_features=5000, sublinear_tf=True),
    MultinomialNB(), "tf-idf + naive Bayes")
results["tf-idf + linear SVM"] = score(
    TfidfVectorizer(token_pattern=r"\w+", min_df=3, max_features=5000, sublinear_tf=True),
    LinearSVC(), "tf-idf + linear SVM")

## Read the ± before the mean

The standard deviation across folds is the part people omit. If two settings
differ by 0.01 and each varies by 0.02 across folds, they are the same setting.

Reporting the better one as an improvement is how a result fails to replicate.

In [ ]:
fig, ax = plt.subplots(figsize=(9, 4.5))
names = list(results)
means = [results[n][0] for n in names]
stds = [results[n][1] for n in names]
ax.barh(names, means, xerr=stds, capsize=4)
ax.set_xlim(min(means) - 0.1, 1.0)
ax.set_xlabel("cross-validated accuracy (error bars = SD across folds)")
ax.set_title("Model choice matters less than you would expect")
plt.tight_layout(); plt.show()

In [ ]:
# Word n-grams: does context between adjacent words help?
for ngram in [(1, 1), (1, 2), (1, 3)]:
    score(TfidfVectorizer(token_pattern=r"\w+", min_df=3, max_features=8000,
                          sublinear_tf=True, ngram_range=ngram),
          LogisticRegression(max_iter=2000), f"tf-idf, n-grams {ngram}")

In [ ]:
# Character n-grams: no tokenising at all. Note what this means for
# African languages — it needs no stopword list, no lemmatiser, no tokeniser.
for lang in ("English", "Hausa", "Yoruba"):
    s = df[df["language"] == lang]
    k = s["category"].value_counts()
    s = s[s["category"].isin(k[k >= 30].index)]
    if s["category"].nunique() < 2:
        continue
    pipe = make_pipeline(
        TfidfVectorizer(analyzer="char_wb", ngram_range=(3, 5), min_df=3,
                        max_features=20000, sublinear_tf=True),
        LogisticRegression(max_iter=2000))
    sc = cross_val_score(pipe, s["text"], s["category"], cv=3, n_jobs=1)
    base = s["category"].value_counts(normalize=True).iloc[0]
    print(f"{lang:<10} char 3-5grams: {sc.mean():.3f}  (baseline {base:.3f})")

## Why that last cell matters more than the rest

Character n-grams work on the raw string. No tokeniser, no stopword list, no
lemmatiser — none of the resources that do not exist for most African languages.

For a language with no NLP toolkit, character n-grams are frequently the strongest
method available and cost nothing to try. That is a practical recommendation you
can act on tomorrow, and it follows directly from Module 03.

### 🔧 Your turn 1

Add one more configuration of your own — a different `max_features`, a different
`min_df`, or `analyzer="char"` instead of `"char_wb"`.

Does it beat the others by more than the error bars? Usually not. That is the
lesson.

### 🔧 Your turn 2

Run the character-n-gram comparison across every language with enough documents.

Where the gap between word-based and character-based methods is largest, you have
identified a language where standard tooling is failing hardest. That map is a
publishable object in itself.

<details markdown="1">
<summary><b>What you should have seen</b> — click to expand</summary>

**Your turn 1.** Almost nothing beats a sensible TF-IDF plus a linear model by
more than the fold-to-fold variation. This is well known and consistently
under-reported: for document classification on modest corpora, the ceiling is set
by the data, not the algorithm. Which means your effort belongs in the corpus and
the labels, not in model selection.

**Your turn 2.** Character n-grams tend to hold up better in morphologically rich
languages, because they capture sub-word regularities that word tokens split
apart. Where word-based methods lag furthest behind, you are seeing the cost of
missing tokenisation and lemmatisation resources — measured, per language, on an
openly licensed corpus. That is a concrete contribution to the African-NLP
literature, and it comes out of a notebook you can run in an afternoon.

</details>

## Where this stops

You can compare methods without fooling yourself. Next: **07c**, where the model
stops being a scoreboard and becomes evidence.